In [47]:
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils import spectral_norm

from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BETAS = (0.5,0.999)
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [68]:
BATCH_SIZE = 128
LATENT_DIM = 100
LR = 1e-4
BETAS = (0.5, 0.999)
BETAS_WGAN = (0.0, 0.9)
EPOCHS = 20
LAMBDA_GP = 10
N_CRITIC = 5

In [49]:
train = torch.load("processed/train.pt",weights_only=False)
meta = torch.load("processed/meta.pt",weights_only=False)
images = train["images"]
labels = train["labels"]

label_names = meta["label_names"]

print(images.shape)
print(labels.shape)
print("\nClasses:")
for idx, name in enumerate(label_names):
    print(f"{idx}: {name}")

torch.Size([50000, 3, 32, 32])
torch.Size([50000])

Classes:
0: airplane
1: automobile
2: bird
3: cat
4: deer
5: dog
6: frog
7: horse
8: ship
9: truck


In [50]:
class Dataset(Dataset):

    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

In [51]:
train_dataset = Dataset(images,labels)
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)

print("Dataset Size:", len(train_dataset))
print("Batches:", len(train_loader))

Dataset Size: 50000
Batches: 391


In [52]:
def trainer(G,D,train_loader,criterion,g_optimizer,d_optimizer,latent_dim,device):
    G.train()
    D.train()
    running_g_loss = 0.0
    running_d_loss = 0.0
    pbar = tqdm(train_loader,leave=False,desc="Training")
    
    for real_images, _ in pbar:
        batch_size = real_images.size(0)
        real_images = real_images.to(device)
        real_labels = torch.ones(batch_size,1,device=device)
        fake_labels = torch.zeros(batch_size,1,device=device)


        d_optimizer.zero_grad()
        real_preds = D(real_images)
        real_loss = criterion(real_preds,real_labels)
        
        z = torch.randn(batch_size,latent_dim,device=device)
        fake_images = G(z)
        fake_preds = D(fake_images.detach())
        fake_loss = criterion(fake_preds,fake_labels)
        d_loss = real_loss + fake_loss
        d_loss.backward()
        d_optimizer.step()

        g_optimizer.zero_grad()
        z = torch.randn(batch_size,latent_dim,device=device)
        fake_images = G(z)
        preds = D(fake_images)
        g_loss = criterion(preds,real_labels)
        g_loss.backward()
        g_optimizer.step()
        
        running_g_loss += g_loss.item()
        running_d_loss += d_loss.item()
        
        pbar.set_postfix(
            G_Loss=f"{g_loss.item():.4f}",
            D_Loss=f"{d_loss.item():.4f}"
        )

    epoch_g_loss = (running_g_loss /len(train_loader))
    epoch_d_loss = (running_d_loss /len(train_loader))

    return epoch_g_loss, epoch_d_loss

#### Experiment 1
##### DCGAN + Spectral Normalization

In [54]:
class DCGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim,512,4,1,0,bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512,256,4,2,1,bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256,128,4,2,1,bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128,3,4,2,1,bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        z = z.view(z.size(0),z.size(1),1,1)
        return self.net(z)

In [55]:
class SNDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            spectral_norm(nn.Conv2d(3, 128, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(128, 256, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(256, 512, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(512, 1, 4, 1, 0, bias=False)),
            nn.Sigmoid()
        )

    def forward(self, x):
        out = self.net(x)
        return out.view(-1, 1)

In [56]:
G_sn = DCGenerator(LATENT_DIM).to(device)
D_sn = SNDiscriminator().to(device)

G_sn.load_state_dict(torch.load("checkpoints/dcgan_generator.pth",map_location=device))

g_params = sum(p.numel()for p in G_sn.parameters())
d_params = sum(p.numel()for p in D_sn.parameters())
criterion = nn.BCELoss()
print(f"Generator Parameters: {g_params:,}")
print(f"SNDiscriminator Parameters: {d_params:,}")

gsn_optimizer = optim.Adam(G_sn.parameters(),lr=LR,betas=BETAS)
dsn_optimizer = optim.Adam(D_sn.parameters(),lr=LR,betas=BETAS)

Generator Parameters: 3,448,576
SNDiscriminator Parameters: 2,635,776


In [57]:
g_losses_sn = []
d_losses_sn = []

for epoch in tqdm(range(EPOCHS), desc="SN-GAN Fine-tuning"):
    g_loss, d_loss = trainer(G_sn,D_sn,train_loader,criterion,gsn_optimizer,dsn_optimizer,LATENT_DIM,device)

    g_losses_sn.append(g_loss)
    d_losses_sn.append(d_loss)

    print(f"Epoch [{epoch+1}/20] | G Loss: {g_loss:.4f} | D Loss: {d_loss:.4f}")
    
torch.save(G_sn.state_dict(),"checkpoints/sn_generator.pth")
torch.save(D_sn.state_dict(),"checkpoints/sn_discriminator.pth")

SN-GAN Fine-tuning:   0%|          | 0/20 [00:00<?, ?it/s]

Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch [1/20] | G Loss: 0.8157 | D Loss: 1.3448


Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch [2/20] | G Loss: 0.8367 | D Loss: 1.2953


...

Epoch [18/20] | G Loss: 0.7238 | D Loss: 1.3550


Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch [19/20] | G Loss: 0.7232 | D Loss: 1.3538


Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch [20/20] | G Loss: 0.7182 | D Loss: 1.3569


#### Experiment 2
##### DCGAN + Extra Layer + Dropout

In [60]:
class ImprovedDiscriminator(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.25),

            nn.Conv2d(128, 256, 3, 1, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.25),

            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()

        )

    def forward(self, x):
        out = self.net(x)
        return out.view(-1,1)

In [61]:
G_drop = DCGenerator(LATENT_DIM).to(device)
D_drop = SNDiscriminator().to(device)

G_drop.load_state_dict(torch.load("checkpoints/dcgan_generator.pth",map_location=device))

g_params = sum(p.numel()for p in G_drop.parameters())
d_params = sum(p.numel()for p in D_drop.parameters())
criterion = nn.BCELoss()
print(f"Generator Parameters: {g_params:,}")
print(f"Dropout Discriminator Parameters: {d_params:,}")

g_drop_optimizer = optim.Adam(G_drop.parameters(),lr=LR,betas=BETAS)
d_drop_optimizer = optim.Adam(D_drop.parameters(),lr=LR,betas=BETAS)

Generator Parameters: 3,448,576
Dropout Discriminator Parameters: 2,635,776


In [62]:
g_losses_drop = []
d_losses_drop = []

for epoch in tqdm(range(EPOCHS), desc="DCGAN+Dropout Fine-tuning"):
    g_loss, d_loss = trainer(G_drop,D_drop,train_loader,criterion,g_drop_optimizer,d_drop_optimizer,LATENT_DIM,device)

    g_losses_drop.append(g_loss)
    d_losses_drop.append(d_loss)

    print(f"Epoch [{epoch+1}/20] | G Loss: {g_loss:.4f} | D Loss: {d_loss:.4f}")
    
torch.save(G_drop.state_dict(),"checkpoints/dcgan_drop_generator.pth")
torch.save(D_drop.state_dict(),"checkpoints/dcgan_drop_discriminator.pth")

DCGAN+Dropout Fine-tuning:   0%|          | 0/20 [00:00<?, ?it/s]

Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch [1/20] | G Loss: 0.8234 | D Loss: 1.3458


Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch [2/20] | G Loss: 0.8591 | D Loss: 1.2753


...

Epoch [18/20] | G Loss: 0.7272 | D Loss: 1.3431


Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch [19/20] | G Loss: 0.7284 | D Loss: 1.3376


Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch [20/20] | G Loss: 0.7218 | D Loss: 1.3453


#### Experiment 3
##### WGAN-GP

In [64]:
class Critic(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3,64,4,2,1,bias=False),
            nn.LeakyReLU(0.2,True),

            nn.Conv2d(64,128,4,2,1,bias=False),
            nn.LeakyReLU(0.2,True),

            nn.Conv2d(128,256,4,2,1,bias=False),
            nn.LeakyReLU(0.2,True),

            nn.Conv2d(256,1,4,1,0,bias=False)
        )

    def forward(self,x):
        return self.net(x).view(-1)

In [69]:
G_wg = DCGenerator(LATENT_DIM).to(device)
C = Critic().to(device)

G_wg.load_state_dict(torch.load("checkpoints/dcgan_generator.pth",map_location=device))

g_params = sum(p.numel()for p in G_wg.parameters())
c_params = sum(p.numel()for p in C.parameters())

print(f"Generator Parameters: {g_params:,}")
print(f"SNDiscriminator Parameters: {d_params:,}")

gwg_optimizer = optim.Adam(G_wg.parameters(),lr=LR,betas=BETAS_WGAN)
c_optimizer = optim.Adam(C.parameters(),lr=LR,betas=BETAS_WGAN)

Generator Parameters: 3,448,576
SNDiscriminator Parameters: 2,635,776


In [70]:
def gradient_penalty(critic,real,fake,device):
    batch_size = real.size(0)
    epsilon = torch.rand(batch_size,1,1,1,device=device)
    
    interpolated = (epsilon*real +(1-epsilon)*fake)
    interpolated.requires_grad_(True)
    mixed_scores = critic(interpolated)
    gradient = torch.autograd.grad(
        outputs=mixed_scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True
    )[0]

    gradient = gradient.view(gradient.size(0),-1)
    gradient_norm = gradient.norm(2,dim=1)
    gp = ((gradient_norm-1)**2).mean()

    return gp

In [72]:
def trainer_wgan(G,C,train_loader,g_optimizer,c_optimizer,latent_dim,device):
    G.train()
    C.train()
    running_g_loss = 0.0
    running_c_loss = 0.0
    pbar = tqdm(train_loader, leave=False, desc="Training")
    for real_images, _ in pbar:
        batch_size = real_images.size(0)
        real_images = real_images.to(device)

        # Train Critic
        c_loss_epoch = 0
        for _ in range(N_CRITIC):
            z = torch.randn(batch_size,latent_dim,device=device)
            fake_images = G(z)
            real_scores = C(real_images)
            fake_scores = C(fake_images.detach())
            gp = gradient_penalty(C,real_images,fake_images.detach(),device)
            c_loss = (-(real_scores.mean())+ fake_scores.mean()+ LAMBDA_GP * gp)
            c_optimizer.zero_grad()
            c_loss.backward()
            c_optimizer.step()
            c_loss_epoch += c_loss.item()

        # Train Generator
        z = torch.randn(batch_size,latent_dim,device=device)
        fake_images = G(z)
        fake_scores = C(fake_images)
        g_loss = -fake_scores.mean()
        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()
        running_g_loss += g_loss.item()
        running_c_loss += c_loss_epoch / N_CRITIC
        pbar.set_postfix(G_Loss=f"{g_loss.item():.4f}",C_Loss=f"{(c_loss_epoch/N_CRITIC):.4f}")
    
    epoch_g_loss = (running_g_loss /len(train_loader))
    epoch_c_loss = (running_c_loss /len(train_loader))

    return epoch_g_loss, epoch_c_loss

In [73]:
g_losses = []
c_losses = []
for epoch in tqdm(range(EPOCHS),desc="WGAN-GP"):
    g_loss,c_loss = trainer_wgan(G_wg,C,train_loader,gwg_optimizer,c_optimizer,LATENT_DIM,device)
    g_losses.append(g_loss)
    c_losses.append(c_loss)
    print(f"Epoch {epoch+1} | G {g_loss:.4f} | C {c_loss:.4f}")
    
torch.save(G_wg.state_dict(),"checkpoints/wgan_generator.pth")
torch.save(C.state_dict(),"checkpoints/wgan_critic.pth")

WGAN-GP:   0%|          | 0/20 [00:00<?, ?it/s]

Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 1 | G 0.4273 | C 0.2377


Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 2 | G 0.2910 | C -0.1852


...

Epoch 18 | G 0.3037 | C -0.4492


Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 19 | G 0.2436 | C -0.5164


Training:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 20 | G 0.4222 | C -0.5332
